# B4 — Option-market tests: informed trading, option returns, and why

Three supplementary tests of whether retail investors "take the right side"
when trading options around earnings announcements, in the spirit of
Bryzgalova, Pavlova & Sikorskaya (2023) and de Silva, So & Smith:

| Test | Question | Module |
|---|---|---|
| 1 | Does pre-announcement option flow predict the announcement return? | `informed_trading` |
| 2 | What did the options each group traded earn across the announcement, before and after trading costs? | `option_returns` |
| 3 | Why — how much of the price change is direction, the move's size, the IV crush, and time decay? | `option_decomposition` |

**Requires A1, A2 and A3.** Retail is CBOE's customer classification;
professional customers are the comparison group throughout. Results are
reported pooled over 2016–2025 and for the three eras used in B3.

**Two design points carried through all three tests.** Day 0 is the
*effective* announcement day: announcements at or after 16:00 ET move day 0
to the next trading day, so pre-announcement windows never contain
post-news trading. And every comparison of retail with professionals is
reported alongside each group's own result, because the groups trade
different firms and different contracts.

## Setup

In [1]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        if (current / "src").exists():
            return current / "src"
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

sys.path.append(str(find_src_dir()))

import polars as pl
from paths import DATA_DIR, IBES_DIR, CRSP_DIR

## Test 1 — Is pre-announcement option flow informed?

    CAR[0,+1] = a + b1·RetailNetPutFlow[−2,−1] + b2·ProNetPutFlow[−2,−1] + controls + year-quarter FE

Net put flow is put buys minus put sells over the two days before day 0,
scaled by the group's total option volume. The announcement return is the
sum of daily returns minus the constructed value-weighted market return over
days 0 and +1. Controls: log market cap, pre-event return (days −20 to −3),
dispersion and log option volume. Standard errors are clustered by firm and
by day-0 date. Flow is standardised, so each coefficient is the change in the
two-day return for a one-standard-deviation increase in flow.

**Reading it:** b1 < 0 means net put buying anticipates bad news (informed,
Pan & Poteshman 2006); b1 ≈ 0 means the flow carries no information.

`sample="both"` requires both groups' flow (Andrew's specification; ~31,000
mostly larger firms). `sample="retail"` uses every event with retail flow
(~94,000). The SUE rows repeat the test with the earnings surprise as the
dependent variable.

Check the printed timing line first: it should read roughly 43% before-open
and 55% after-close. A before-open share near zero means the announcement
times were misparsed, and the function prints a warning if it falls below 15%.

In [2]:
from analysis.informed_trading import build_event_panel, informed_trading_table

panel = build_event_panel()
tbl1 = informed_trading_table(panel)
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=220):
    print(tbl1)

Events 2016-01-01 .. 2025-12-31: 146,523
  dropped 194 whose flow window falls in a CBOE data gap
  with a complete flow window: 146,329  |  linked to a PERMNO: 140,817
  with a complete CAR[0,+1]: 140,601
  announcement timing: after_close 78,180, before_open 60,360, during_market 2,264, unknown 13
  events with retail option volume in [-2,-1]: 94,039
  events with procust option volume in [-2,-1]: 31,141
shape: (16, 11)
┌─────────────────┬─────────┬────────┬────────────┬───────┬──────────┬───────────┬──────────┬───────────┬────────────┬───────────┐
│ dv              ┆ flow    ┆ sample ┆ era        ┆ n     ┆ b_retail ┆ se_retail ┆ p_retail ┆ b_procust ┆ se_procust ┆ p_procust │
│ ---             ┆ ---     ┆ ---    ┆ ---        ┆ ---   ┆ ---      ┆ ---       ┆ ---      ┆ ---       ┆ ---        ┆ ---       │
│ str             ┆ str     ┆ str    ┆ str        ┆ i64   ┆ f64      ┆ f64       ┆ f64      ┆ f64       ┆ f64        ┆ f64       │
╞═════════════════╪═════════╪════════╪════════════

### Test 1 robustness

Three checks on the retail result:

1. **Flow definition.** `pp_ratio` is Pan & Poteshman's opening-buy put ratio
   (purchases only); `net_bearish` is net put buying minus net call buying.
2. **Firm size** (terciles of log market cap).
3. **Professional presence:** events with and without professional customer
   flow. The retail-only and both-groups samples differ in firm size *and* in
   whether professionals trade, so this separates the two. The gap between
   subsets is not formally tested here; an interaction term would be needed
   to claim it.

In [3]:
rob = informed_trading_table(panel, dvs=("car01",),
                             flows=("net_put", "pp_ratio", "net_bearish"), samples=("retail",))

sized = panel.with_columns(pl.col("log_mktcap").qcut(3, labels=["small", "mid", "large"]).alias("size_t"))
size_tbl = pl.concat([
    informed_trading_table(sized.filter(pl.col("size_t") == s), dvs=("car01",), samples=("retail",))
    .with_columns(pl.lit(s).alias("size"))
    for s in ["small", "mid", "large"]])

pro_tbl = pl.concat([
    informed_trading_table(sub, dvs=("car01",), samples=("retail",)).with_columns(pl.lit(label).alias("subset"))
    for label, sub in [("with_pro", panel.filter(pl.col("procust_net_put").is_not_null())),
                       ("no_pro", panel.filter(pl.col("procust_net_put").is_null()))]])

with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=200):
    for name, t in [("flow measures", rob), ("size terciles", size_tbl), ("professional presence", pro_tbl)]:
        print(f"\n=== {name} ===")
        print(t.select([c for c in ["flow", "size", "subset", "era", "n", "b_retail", "se_retail", "p_retail"]
                        if c in t.columns]))


=== flow measures ===
shape: (12, 6)
┌─────────────┬────────────┬───────┬──────────┬───────────┬──────────┐
│ flow        ┆ era        ┆ n     ┆ b_retail ┆ se_retail ┆ p_retail │
│ ---         ┆ ---        ┆ ---   ┆ ---      ┆ ---       ┆ ---      │
│ str         ┆ str        ┆ i64   ┆ f64      ┆ f64       ┆ f64      │
╞═════════════╪════════════╪═══════╪══════════╪═══════════╪══════════╡
│ net_bearish ┆ pooled     ┆ 94000 ┆ -0.0007  ┆ 0.0003    ┆ 0.0419   │
│ net_bearish ┆ pre_covid  ┆ 34857 ┆ -0.0021  ┆ 0.0005    ┆ 0.0000   │
│ net_bearish ┆ covid      ┆ 19476 ┆ -0.0007  ┆ 0.0006    ┆ 0.2748   │
│ net_bearish ┆ post_covid ┆ 34504 ┆ 0.0007   ┆ 0.0007    ┆ 0.3132   │
│ net_put     ┆ pooled     ┆ 94000 ┆ -0.0007  ┆ 0.0003    ┆ 0.0233   │
│ net_put     ┆ pre_covid  ┆ 34857 ┆ -0.0016  ┆ 0.0004    ┆ 0.0003   │
│ net_put     ┆ covid      ┆ 19476 ┆ -0.0006  ┆ 0.0006    ┆ 0.3461   │
│ net_put     ┆ post_covid ┆ 34504 ┆ 0.0002   ┆ 0.0006    ┆ 0.7736   │
│ pp_ratio    ┆ pooled     ┆ 84206 ┆ -0

## Test 2 — What did the options earn across the announcement?

For every contract a group traded on day −1, the holding-period return to
day +1 at OptionMetrics bid-ask midpoints (15:59 ET). Contracts expiring
inside the window are valued at their payoff. Measures, each from the
group's own side:

- `vw_open_buy` / `vw_buy` — volume-weighted mean return on contracts opened
  long / bought (Andrew's specification)
- `dw_buy` — dollar gain per dollar of premium on what the group bought: the
  headline measure, and not dominated by very cheap contracts
- `dw_short` — the group's own return on contracts it sold
- `dw_net` — return on the group's net position

Group means carry firm-and-date clustered standard errors; `diff` compares
the two groups within the same events. **Read the medians alongside the
means:** the returns are strongly right-skewed (most positions lose, a few
pay off hugely), so the mean alone misstates the typical outcome.

In [4]:
from analysis.option_returns import PANEL_PATH, event_option_returns, option_returns_table, spread_comparison

contracts = pl.read_parquet(PANEL_PATH)
tbl2 = option_returns_table(event_option_returns(contracts))
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    print(tbl2.select("measure", "era", "retail_n", "retail_mean", "retail_p", "retail_median",
                      "procust_n", "procust_mean", "procust_median", "diff_n", "diff_mean", "diff_p"))

shape: (20, 12)
┌─────────────┬────────────┬──────────┬─────────────┬──────────┬───────────────┬───────────┬──────────────┬────────────────┬────────┬───────────┬────────┐
│ measure     ┆ era        ┆ retail_n ┆ retail_mean ┆ retail_p ┆ retail_median ┆ procust_n ┆ procust_mean ┆ procust_median ┆ diff_n ┆ diff_mean ┆ diff_p │
│ ---         ┆ ---        ┆ ---      ┆ ---         ┆ ---      ┆ ---           ┆ ---       ┆ ---          ┆ ---            ┆ ---    ┆ ---       ┆ ---    │
│ str         ┆ str        ┆ i64      ┆ f64         ┆ f64      ┆ f64           ┆ i64       ┆ f64          ┆ f64            ┆ i64    ┆ f64       ┆ f64    │
╞═════════════╪════════════╪══════════╪═════════════╪══════════╪═══════════════╪═══════════╪══════════════╪════════════════╪════════╪═══════════╪════════╡
│ vw_open_buy ┆ pooled     ┆ 70991    ┆ 0.0622      ┆ 0.0000   ┆ -0.2703       ┆ 13197     ┆ -0.0276      ┆ -0.3939        ┆ 12907  ┆ 0.0144    ┆ 0.1620 │
│ vw_open_buy ┆ pre_covid  ┆ 27499    ┆ 0.0560      ┆ 

### Test 2 after trading costs

The midpoint returns exclude the bid-ask spread. `cost_frac` is the fraction
of the quoted half-spread paid on each trade: 0 is the midpoint, 0.5 a
realistic case given price improvement on retail orders, 1 buying at the ask
and selling at the bid (an upper bound). Contracts settled at expiry pay no
exit spread.

Both groups are charged the same fraction. Professional customers probably
execute better, so the **post-cost comparison between the groups is not
conclusive**; the robust result is each group's own return.

In [5]:
cols = ["cost_frac", "measure", "era", "retail_mean", "retail_median", "procust_mean",
        "procust_median", "diff_n", "diff_mean", "diff_p"]
cmp_all = spread_comparison(contracts, measures=("dw_buy", "dw_net"))
cmp_otm = spread_comparison(contracts, measures=("dw_buy", "dw_net"), moneyness="otm")
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=250):
    print("=== all contracts ==="); print(cmp_all.select(cols))
    print("\n=== OTM only ===");   print(cmp_otm.select(cols))

=== all contracts ===
shape: (24, 10)
┌───────────┬─────────┬────────────┬─────────────┬───────────────┬──────────────┬────────────────┬────────┬───────────┬────────┐
│ cost_frac ┆ measure ┆ era        ┆ retail_mean ┆ retail_median ┆ procust_mean ┆ procust_median ┆ diff_n ┆ diff_mean ┆ diff_p │
│ ---       ┆ ---     ┆ ---        ┆ ---         ┆ ---           ┆ ---          ┆ ---            ┆ ---    ┆ ---       ┆ ---    │
│ f64       ┆ str     ┆ str        ┆ f64         ┆ f64           ┆ f64          ┆ f64            ┆ i64    ┆ f64       ┆ f64    │
╞═══════════╪═════════╪════════════╪═════════════╪═══════════════╪══════════════╪════════════════╪════════╪═══════════╪════════╡
│ 0.0000    ┆ dw_buy  ┆ pooled     ┆ 0.0382      ┆ -0.1957       ┆ -0.0177      ┆ -0.3065        ┆ 14462  ┆ 0.0023    ┆ 0.7824 │
│ 0.0000    ┆ dw_buy  ┆ pre_covid  ┆ 0.0403      ┆ -0.1966       ┆ -0.0170      ┆ -0.2788        ┆ 3285   ┆ 0.0308    ┆ 0.1086 │
│ 0.0000    ┆ dw_buy  ┆ covid      ┆ -0.0170     ┆ -0.1932 

### Why costs bite harder for retail

The quoted spread as a share of premium, weighted by contracts bought. This
figure is pooled and dollar-weighted, so large liquid names dominate it; the
event-level means above weight every firm-event equally and so reflect the
wider spreads of smaller firms. The ratio between the groups is the robust
part.

In [6]:
priced = contracts.filter(pl.col("ret").is_not_null())
for g in ("retail", "procust"):
    b = pl.col(f"{g}_open_buy") + pl.col(f"{g}_close_buy")
    rel = priced.select((b * (pl.col("ask_entry") - pl.col("bid_entry"))).sum()
                        / (b * pl.col("mid_entry")).sum()).item()
    print(f"{g}: quoted spread = {rel:.1%} of premium, weighted by contracts bought")

retail: quoted spread = 6.8% of premium, weighted by contracts bought
procust: quoted spread = 3.5% of premium, weighted by contracts bought


## Test 3 — Why do the options lose value?

Each contract's midpoint price change from day −1 to day +1, decomposed with
its day −1 Greeks:

    dP = Delta·dS + ½·Gamma·dS² + Vega·dσ + Theta·(days/365) + residual
         direction   convexity     IV change   time decay

Units follow the IvyDB US manual: Vega is $ per 1.00 change in implied
volatility (cents per volatility point), so it multiplies the IV change as a
decimal; Theta is $ per year over calendar days. Contracts that expired
inside the window (no exit IV) and contracts with missing IV or Greeks are
excluded, with their share of volume printed.

**Reading it:** gamma and theta are two halves of one bet — together they
measure whether the realised move beat the move the option price expected.
The component means add up exactly to the total; the medians do not and are
read one term at a time. Check the fit line first: an R² around 0.97 and a
small median residual mean the decomposition can be trusted.

In [7]:
from analysis.option_decomposition import (decomposition_panel, event_decomposition,
                                           decomposition_table, right_direction_summary)

d = decomposition_panel(contracts)
tbl3 = decomposition_table(event_decomposition(d))
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=220):
    print(tbl3.filter(pl.col("era") == "pooled").select("group", "component", "n", "mean", "se", "median"))
    print(tbl3.filter(pl.col("component") == "IV change (vega)").select("group", "era", "mean", "median"))

Priced contract-events: 2,384,569  |  decomposable: 1,952,305 (81.9%)
  retail buy volume: 68.8% decomposable, 12.5% expired inside the window, 18.7% missing IV/Greeks
  procust buy volume: 74.1% decomposable, 10.1% expired inside the window, 15.8% missing IV/Greeks
  fit: R^2 of actual on approximated change 0.974; median |residual| = 7.9% of premium
shape: (12, 6)
┌─────────┬───────────────────────┬───────┬─────────┬────────┬─────────┐
│ group   ┆ component             ┆ n     ┆ mean    ┆ se     ┆ median  │
│ ---     ┆ ---                   ┆ ---   ┆ ---     ┆ ---    ┆ ---     │
│ str     ┆ str                   ┆ i64   ┆ f64     ┆ f64    ┆ f64     │
╞═════════╪═══════════════════════╪═══════╪═════════╪════════╪═════════╡
│ retail  ┆ direction (delta)     ┆ 71383 ┆ -0.0397 ┆ 0.0039 ┆ -0.0145 │
│ retail  ┆ convexity (gamma)     ┆ 71383 ┆ 0.3353  ┆ 0.0047 ┆ 0.0995  │
│ retail  ┆ IV change (vega)      ┆ 71383 ┆ -0.1336 ┆ 0.0038 ┆ -0.1719 │
│ retail  ┆ time decay (theta)    ┆ 71383 ┆ -0.

### Right direction, still lost?

Andrew's hypothesis: even when retail calls the direction correctly, the IV
crush often wipes out the gain. Among positions whose delta term was positive
(direction right), weighted by contracts bought:

- `share_right` — share of volume with the direction right
- `right_but_lost` — of those, the share that still lost money
- `iv_outweighed` — of those, the share where the IV change alone exceeded
  the directional gain (delta plus gamma)
- `right_*` — dollar-weighted components for the right-direction positions

In [8]:
cols = ["group", "era", "share_right", "right_but_lost", "iv_outweighed",
        "right_delta_term", "right_vega_term", "right_dP"]
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=4, tbl_width_chars=220):
    print("=== all contracts ==="); print(right_direction_summary(d).select(cols))
    print("\n=== OTM only ===");   print(right_direction_summary(d, moneyness="otm").select(cols))

=== all contracts ===
shape: (8, 8)
┌─────────┬────────────┬─────────────┬────────────────┬───────────────┬──────────────────┬─────────────────┬──────────┐
│ group   ┆ era        ┆ share_right ┆ right_but_lost ┆ iv_outweighed ┆ right_delta_term ┆ right_vega_term ┆ right_dP │
│ ---     ┆ ---        ┆ ---         ┆ ---            ┆ ---           ┆ ---              ┆ ---             ┆ ---      │
│ str     ┆ str        ┆ f64         ┆ f64            ┆ f64           ┆ f64              ┆ f64             ┆ f64      │
╞═════════╪════════════╪═════════════╪════════════════╪═══════════════╪══════════════════╪═════════════════╪══════════╡
│ retail  ┆ pooled     ┆ 0.4729      ┆ 0.4590         ┆ 0.3016        ┆ 0.7106           ┆ -0.3333         ┆ 0.5782   │
│ retail  ┆ pre_covid  ┆ 0.4819      ┆ 0.4331         ┆ 0.3122        ┆ 0.7374           ┆ -0.3419         ┆ 0.6378   │
│ retail  ┆ covid      ┆ 0.4654      ┆ 0.4803         ┆ 0.2922        ┆ 0.6446           ┆ -0.3014         ┆ 0.4648   │
│ re

## Write tables and figures to `results/`

Everything reported from Tests 1–3 is written to disk, so nothing in the
thesis is copied from notebook output:

- `table7_informed_trading.csv`, `table7b_informed_trading_robustness.csv`
- `table8_option_returns.csv`, `table9_option_returns_costs.csv`
- `table10_loss_decomposition.csv`, `table11_right_direction.csv`
- `figures/fig_informed_flow_by_era.png`, `fig_option_return_distribution.png`,
  `fig_option_returns_after_costs.png`, `fig_option_loss_decomposition.png`,
  `fig_option_right_direction.png`

Figure titles are descriptive; the interpretation belongs in the thesis
captions. This recomputes the tables above, so allow several minutes.

In [9]:
from analysis.option_market_results import build_option_market_results

results = build_option_market_results(panel, contracts, d=d)

Building option-market results...
  wrote table7_informed_trading.csv  (16 rows)
  wrote table7b_informed_trading_robustness.csv  (32 rows)
  wrote table8_option_returns.csv  (20 rows)
  wrote table9_option_returns_costs.csv  (48 rows)
  wrote table10_loss_decomposition.csv  (48 rows)
  wrote table11_right_direction.csv  (16 rows)
  wrote figures/fig_informed_flow_by_era.png
  wrote figures/fig_option_return_distribution.png
  wrote figures/fig_option_returns_after_costs.png
  wrote figures/fig_option_loss_decomposition.png
  wrote figures/fig_option_right_direction.png
done


## Summary of findings

Figures from the run of 25 September 2026; a re-run on the same data
reproduces them.

**Test 1 — no directional skill in put buying.** Pan & Poteshman's
opening-buy put ratio has no predictive power for the announcement return in
any era. Retail's *net* positioning, which includes put writing, carried a
small but significant signal before 2020: a one-standard-deviation increase
in net put flow predicted a two-day return 16 bp lower (21 bp for net
bearish flow; p < 0.001). The signal sits in mid and large firms, is
concentrated in events without professional customer flow, and disappears
from 2020 onward. Professional put buying predicts earnings surprises but not
returns.

**Test 2 — break-even at the midpoint, clear losses after costs.** At
midpoints, the options retail buys roughly break even on average (+3.8% per
dollar) while the typical position loses about 20% (about 31% for
out-of-the-money contracts). Charging half the quoted spread turns the
average into −14% (−18% for OTM), and the full quoted spread into −28%
(−35%). Retail's contracts carry about twice professionals' relative spread
(6.8% against 3.5% of premium), so costs bite harder for retail.

**Test 3 — the IV crush sinks the typical position.** The decomposition fits
well (R² 0.974; median residual 7.9% of premium) and covers 69% of retail's
buy volume. Direction contributes almost nothing (median −1.5%). The typical
loss comes from the IV crush (median −17%), with the announcement move
roughly paying for time decay. Professionals lose more to the IV crush (about
−20%) because they hold more vega-heavy positions, which explains their
larger typical losses in Test 2. Retail calls the direction right on 47–49%
of volume, and more than half (52.5%) of its correctly called OTM positions
still lost money; in about a third, the IV crush alone outweighed the
directional gain.

**Together.** Retail traders show no directional edge ahead of earnings. The
options they buy are roughly fairly priced at the midpoint on average but
lose in the typical case to the IV crush, and trading costs — nearly twice
professionals' in relative terms — turn that into a clear loss, consistent
with de Silva, So & Smith. Outcomes worsened in the COVID era and returned
to pre-COVID levels afterwards without improving on them. These are aggregate
data, so they cannot separate whether individual traders learned from a
change in who was trading.